<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
PyTorch Autoencoder for Cyber Security Anomaly Detection
</div>

- Dataset: KDD Cup 1999 SA subset
- 설치: pip install scikit-learn pandas numpy matplotlib torch joblib

In [ ]:
import copy
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from sklearn.datasets import fetch_kddcup99
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 기본 설정

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
device

# KDD99 feature 설명

In [ ]:
# ============================================================
# 1. KDD99 feature 설명
# ============================================================
# row 1개 = 네트워크 connection / flow 1건
# column 1개 = connection을 설명하는 feature
#
# label:
# - normal. = 정상
# - normal. 이외 = 공격 / 이상
#
# 이 예제에서는:
# - 0 = normal
# - 1 = anomaly

feature_descriptions = {
    "duration": "connection 지속 시간, 초 단위",
    "protocol_type": "프로토콜 종류. 예: tcp, udp, icmp",
    "service": "대상 네트워크 서비스. 예: http, smtp, ftp 등",
    "flag": (
        "connection 상태 요약값. TCP header의 raw flag 비트가 아니라 "
        "connection의 정상/오류 상태를 나타냄. 예: SF=정상 연결/종료, "
        "S0=연결 시도 후 응답 없음, REJ=연결 거절"
    ),
    "src_bytes": (
        "connection 단위에서 source에서 destination 방향으로 전송된 data byte 총합. "
        "row 1개는 패킷 1개가 아니라 connection/flow 1개"
    ),
    "dst_bytes": (
        "connection 단위에서 destination에서 source 방향으로 전송된 data byte 총합"
    ),
    "land": "source/destination IP와 port가 모두 같으면 1",
    "wrong_fragment": "잘못된 fragment 수",
    "urgent": "urgent packet 수",
    "hot": "보안상 민감한 동작 indicator 수",
    "num_failed_logins": "로그인 실패 횟수",
    "logged_in": "로그인 성공 여부",
    "num_compromised": "compromised condition 수",
    "root_shell": "root shell 획득 여부",
    "su_attempted": "su root 명령 시도 여부",
    "num_root": "root access 수",
    "num_file_creations": "파일 생성 횟수",
    "num_shells": "shell prompt 획득 횟수",
    "num_access_files": "access control file 접근 횟수",
    "num_outbound_cmds": "outbound command 수",
    "is_host_login": "host login 여부",
    "is_guest_login": "guest login 여부",
    "count": "최근 2초 동안 같은 destination host로의 connection 수",
    "srv_count": "최근 2초 동안 같은 service로의 connection 수",
    "serror_rate": "SYN error connection 비율",
    "srv_serror_rate": "같은 service 대상 SYN error 비율",
    "rerror_rate": "REJ error connection 비율",
    "srv_rerror_rate": "같은 service 대상 REJ error 비율",
    "same_srv_rate": "같은 service connection 비율",
    "diff_srv_rate": "다른 service connection 비율",
    "srv_diff_host_rate": "같은 service 중 다른 host로 향한 비율",
    "dst_host_count": "최근 100개 connection 중 같은 destination host 수",
    "dst_host_srv_count": "최근 100개 connection 중 같은 destination host와 service 수",
    "dst_host_same_srv_rate": "destination host 기준 같은 service 비율",
    "dst_host_diff_srv_rate": "destination host 기준 다른 service 비율",
    "dst_host_same_src_port_rate": "destination host 기준 같은 source port 비율",
    "dst_host_srv_diff_host_rate": "destination host/service 기준 다른 host 비율",
    "dst_host_serror_rate": "destination host 기준 SYN error 비율",
    "dst_host_srv_serror_rate": "destination host/service 기준 SYN error 비율",
    "dst_host_rerror_rate": "destination host 기준 REJ error 비율",
    "dst_host_srv_rerror_rate": "destination host/service 기준 REJ error 비율",
}

In [ ]:
flag_descriptions = {
    "SF": "정상적으로 연결되고 정상적으로 종료됨",
    "S0": "연결 시도는 있었지만 응답이 없음",
    "S1": "연결은 되었지만 종료가 관찰되지 않음",
    "S2": "연결 후 originator가 종료를 시도했지만 responder 응답이 없음",
    "S3": "연결 후 responder가 종료를 시도했지만 originator 응답이 없음",
    "REJ": "연결 시도가 거절됨",
    "RSTO": "연결 후 originator가 RST로 강제 종료",
    "RSTR": "responder가 RST로 강제 종료",
    "OTH": "SYN 없이 중간 트래픽만 관찰됨",
}

# 시각화 함수

In [ ]:
def plot_label_distribution(y_values, title):
    counts = pd.Series(y_values).value_counts().reindex([0, 1], fill_value=0)

    plt.figure(figsize=(6, 4))
    plt.bar(["normal", "anomaly"], counts.values)
    plt.title(title)
    plt.xlabel("Label")
    plt.ylabel("Count")

    for i, v in enumerate(counts.values):
        plt.text(i, v, f"{v:,}", ha="center", va="bottom")

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_loss_history(train_losses, val_losses):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="train normal loss")
    plt.plot(val_losses, label="validation normal loss")
    plt.title("Autoencoder Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_confusion_matrix(cm, title):
    labels = ["normal", "anomaly"]

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.xticks([0, 1], labels)
    plt.yticks([0, 1], labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_latent_space_2d(Z, y_true, title):
    """
    Encoder의 latent vector 중 앞의 2개 차원을 사용해 시각화합니다.

    latent_dim이 2 이상이어야 합니다.
    이 그림은 모델 성능 평가용이라기보다,
    encoder가 만든 저차원 표현을 이해하기 위한 용도입니다.
    """
    if Z.shape[1] < 2:
        print("latent_dim이 2보다 작아서 2D 시각화를 할 수 없습니다.")
        return

    y_true = np.asarray(y_true)

    plt.figure(figsize=(8, 6))
    plt.scatter(
        Z[y_true == 0, 0],
        Z[y_true == 0, 1],
        s=8,
        alpha=0.5,
        label="normal"
    )
    plt.scatter(
        Z[y_true == 1, 0],
        Z[y_true == 1, 1],
        s=8,
        alpha=0.8,
        label="anomaly"
    )
    plt.title(title)
    plt.xlabel("latent_0")
    plt.ylabel("latent_1")
    plt.legend()
    plt.tight_layout()
    plt.show()

# 데이터 다운로드

In [ ]:
kdd = fetch_kddcup99(
    subset="SA",
    percent10=True,
    as_frame=True,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [ ]:
X = kdd.data.copy()
y_raw = pd.Series(kdd.target)

In [ ]:
# label이 bytes 형태일 수 있으므로 문자열로 변환
y_raw = y_raw.map(lambda v: v.decode("utf-8") if isinstance(v, bytes) else str(v))

In [ ]:
# normal. 이면 0, 그 외는 1
y = (y_raw != "normal.").astype(int)

In [ ]:
# object 컬럼의 bytes 값을 문자열로 변환
for col in X.select_dtypes(include=["object"]).columns:
    X[col] = X[col].map(lambda v: v.decode("utf-8") if isinstance(v, bytes) else v)

# 데이터 로딩 직후 실제 데이터 확인

In [ ]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

In [ ]:
# 데이터 크기
print("X shape:", X.shape)
print("y shape:", y.shape)
print("row 개수:", f"{X.shape[0]:,}")
print("feature 개수:", X.shape[1])

In [ ]:
# 실제 데이터 앞부분 10개
X.head(10)

In [ ]:
# label 앞부분 10개
pd.DataFrame({
    "original_label": y_raw.head(10),
    "binary_label": y.head(10)
})

In [ ]:
# 정상 / 이상 분포
print(y.value_counts().rename({0: "normal", 1: "anomaly"}))
print("---")
print(y.value_counts(normalize=True).rename({0: "normal", 1: "anomaly"}))

In [ ]:
# feature 설명
for i, col in enumerate(X.columns, start=1):
    print(f"{i:02d}. {col}: {feature_descriptions.get(col, '설명 미등록')}")

In [ ]:
# flag 값 의미
for k, v in flag_descriptions.items():
    print(f"{k}: {v}")

In [ ]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Categorical features:", categorical_cols)
print("Numeric feature count:", len(numeric_cols))

In [ ]:
# category feature 값 확인
for col in categorical_cols:
    print(f"\n[{col}] unique 값 개수:", X[col].nunique())
    print(X[col].value_counts())

In [ ]:
plot_label_distribution(y, "Initial Normal vs Anomaly Count")

# Train / Validation / Test 분리

In [ ]:
# 전체 데이터:
# - train      : 60%
# - validation : 20%
# - test       : 20%
#
# y_raw도 같이 나누는 이유:
# - 모델 학습에는 쓰지 않음
# - 나중에 어떤 공격 유형인지 사후 분석하기 위해 보관

In [ ]:
X_temp, X_test, y_temp, y_test, y_raw_temp, y_raw_test = train_test_split(
    X,
    y,
    y_raw,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

In [ ]:
X_train_all, X_val, y_train_all, y_val, y_raw_train, y_raw_val = train_test_split(
    X_temp,
    y_temp,
    y_raw_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

In [ ]:
# Train / Validation / Test 분리 결과
print("X_train_all:", X_train_all.shape)
print("X_val      :", X_val.shape)
print("X_test     :", X_test.shape)

In [ ]:
print("Train label 비율")
print(y_train_all.value_counts(normalize=True).rename({0: "normal", 1: "anomaly"}))

print("\nValidation label 비율")
print(y_val.value_counts(normalize=True).rename({0: "normal", 1: "anomaly"}))

print("\nTest label 비율")
print(y_test.value_counts(normalize=True).rename({0: "normal", 1: "anomaly"}))

# 정상 데이터만 학습에 사용

In [ ]:
# Autoencoder는 정상 데이터를 자기 자신으로 복원하도록 학습합니다.
#
# 정상 데이터만 학습했으므로:
# - 정상 데이터는 잘 복원됨
# - 이상 데이터는 잘 복원되지 않을 가능성이 큼
# - reconstruction error가 크면 anomaly로 판단

In [ ]:
X_train_normal = X_train_all[y_train_all == 0]
X_val_normal = X_val[y_val == 0]

In [ ]:
# 실습 속도를 위한 샘플링
max_train_size = 100_000

In [ ]:
if len(X_train_normal) > max_train_size:
    X_train_normal = X_train_normal.sample(
        n=max_train_size,
        random_state=RANDOM_STATE
    )

print("[+] Autoencoder 학습용 정상 데이터")
print("X_train_normal:", X_train_normal.shape)
print("X_val_normal  :", X_val_normal.shape)

# 전처리

In [ ]:
# PyTorch 모델은 숫자 tensor만 입력으로 받을 수 있습니다.
#
# categorical feature:
# - protocol_type, service, flag
# - OneHotEncoder로 숫자화
#
# numeric feature:
# - StandardScaler로 scaling
#
# 중요:
# - preprocessor는 X_train_normal에만 fit합니다.
# - validation/test는 transform만 합니다.
# - 실제 운영에서 미래 데이터를 처리하는 방식과 같습니다.

In [ ]:
onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
preprocess = ColumnTransformer(
    transformers=[
        ("cat", onehot, categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ]
)

In [ ]:
# OneHotEncoder는 train 정상 데이터에서 category 목록을 학습하고, StandardScaler는 train 정상 데이터의 숫자 feature 평균과 표준편차를 학습합니다.
X_train_normal_pre = preprocess.fit_transform(X_train_normal)

In [ ]:
X_val_pre = preprocess.transform(X_val) # 정상과 이상이 모두 섞인 validation 데이터. threshold 결정과 validation 성능 평가에 사용합니다.
X_val_normal_pre = preprocess.transform(X_val_normal) # validation 중 정상 데이터만 모은 것. 학습 중 early stopping용 validation loss에 사용
X_test_pre = preprocess.transform(X_test) # 실제 미래 데이터처럼 예측. 마지막에만 성능 평가

In [ ]:
X_train_normal_pre = X_train_normal_pre.astype("float32")
X_val_pre = X_val_pre.astype("float32")
X_val_normal_pre = X_val_normal_pre.astype("float32")
X_test_pre = X_test_pre.astype("float32")

In [ ]:
# 전처리 후 데이터 크기
print("X_train_normal_pre:", X_train_normal_pre.shape)
print("X_val_normal_pre  :", X_val_normal_pre.shape)
print("X_val_pre         :", X_val_pre.shape)
print("X_test_pre        :", X_test_pre.shape)

In [ ]:
input_dim = X_train_normal_pre.shape[1]
input_dim

# PyTorch Dataset / DataLoader 생성

In [ ]:
batch_size = 256

In [ ]:
train_tensor = torch.tensor(X_train_normal_pre, dtype=torch.float32)
val_normal_tensor = torch.tensor(X_val_normal_pre, dtype=torch.float32)

In [ ]:
train_dataset = TensorDataset(train_tensor, train_tensor)
val_normal_dataset = TensorDataset(val_normal_tensor, val_normal_tensor)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_normal_loader = DataLoader(val_normal_dataset, batch_size=batch_size, shuffle=False)

# Encoder / Decoder / Autoencoder 모델 분리 정의

In [ ]:
# 목적:
# - Autoencoder 전체는 이상탐지 학습에 사용
# - Encoder는 나중에 차원 축소 / feature extraction 용도로 따로 사용
# - Decoder는 latent vector를 원래 feature 공간으로 복원
#
# 구조:
#
# input feature vector
#        ↓
#     Encoder
#        ↓
# latent vector
#        ↓
#     Decoder
#        ↓
# reconstructed input
#
# Autoencoder = Encoder + Decoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()

        self.input_dim = input_dim
        self.latent_dim = latent_dim

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, latent_dim),
            nn.ReLU()
        )

    def forward(self, x):
        z = self.net(x)
        return z

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, latent_dim=16):
        super().__init__()

        self.output_dim = output_dim
        self.latent_dim = latent_dim

        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Linear(32, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        reconstructed = self.net(z)
        return reconstructed

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed

In [ ]:
# latent_dim:
# - Encoder가 만드는 저차원 표현의 차원 수
# - 16이면 고차원 one-hot feature를 16차원 latent vector로 압축
# - 시각화 목적이면 2로 설정할 수도 있음
# - 일반적으로는 8, 16, 32 등을 실험
latent_dim = 16

In [ ]:
encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
decoder = Decoder(output_dim=input_dim, latent_dim=latent_dim).to(device)
autoencoder = Autoencoder(encoder=encoder, decoder=decoder).to(device)

In [ ]:
# Loss function
criterion = nn.MSELoss()

# optimizer는 autoencoder 전체 파라미터를 학습합니다.
# 즉, encoder와 decoder가 함께 학습됩니다.
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)

In [ ]:
# Encoder 구조
print(encoder)

In [ ]:
# Decoder 구조
print(decoder)

In [ ]:
# Autoencoder 구조
print(autoencoder)

# 학습 함수

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_count = 0

    for batch_x, batch_y in data_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        current_batch_size = batch_x.size(0)
        total_loss += loss.item() * current_batch_size
        total_count += current_batch_size

    return total_loss / total_count

In [ ]:
def evaluate_loss(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            current_batch_size = batch_x.size(0)
            total_loss += loss.item() * current_batch_size
            total_count += current_batch_size

    return total_loss / total_count

# 모델 학습

In [ ]:
# Autoencoder 학습:
# - 입력:  X_train_normal_pre
# - 정답:  X_train_normal_pre
#
# 즉, 정상 데이터를 자기 자신으로 복원하도록 학습합니다.
#
# Early stopping:
# - validation normal loss가 좋아지지 않으면 학습 중단
# - 과적합을 줄이는 목적

In [ ]:
epochs = 100
patience = 10

In [ ]:
best_val_loss = float("inf")
best_model_state = None
patience_counter = 0

In [ ]:
train_losses = []
val_losses = []

In [ ]:
for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(
        model=autoencoder,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    val_loss = evaluate_loss(
        model=autoencoder,
        data_loader=val_normal_loader,
        criterion=criterion,
        device=device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch:02d}/{epochs}] "
        f"train_loss={train_loss:.6f} "
        f"val_normal_loss={val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(autoencoder.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

In [ ]:
# 가장 validation loss가 좋았던 autoencoder weight 복원
autoencoder.load_state_dict(best_model_state)

In [ ]:
# autoencoder 안에 들어 있는 encoder, decoder도 함께 복원됩니다.
encoder = autoencoder.encoder
decoder = autoencoder.decoder

In [ ]:
plot_loss_history(train_losses, val_losses)

# Reconstruction error 계산 함수

In [ ]:
# reconstruction error:
# - 원본 입력과 복원 출력의 차이
# - 값이 클수록 복원이 잘 안 됨
# - 정상 패턴에서 벗어났을 가능성이 큼

In [ ]:
def reconstruction_error(model, X_array, device, batch_size=1024):
    model.eval()

    dataset = TensorDataset(torch.tensor(X_array, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    errors = []

    with torch.no_grad():
        for (batch_x,) in loader:
            batch_x = batch_x.to(device)

            outputs = model(batch_x)

            # sample별 MSE
            batch_errors = torch.mean((batch_x - outputs) ** 2, dim=1)

            errors.append(batch_errors.cpu().numpy())

    return np.concatenate(errors)

In [ ]:
train_error = reconstruction_error(autoencoder, X_train_normal_pre, device)
val_error = reconstruction_error(autoencoder, X_val_pre, device)

In [ ]:
print("[+] Train normal reconstruction error 요약")
print(pd.Series(train_error).describe())
print("---")
print("[+] Validation reconstruction error 요약")
print(pd.Series(val_error).describe())

# Validation 데이터로 threshold 결정

In [ ]:
# threshold:
# - reconstruction_error >= threshold 이면 anomaly
# - reconstruction_error < threshold 이면 normal
#
# 여기서는 validation F1-score가 가장 높은 threshold를 선택합니다.

In [ ]:
def find_best_threshold_by_f1(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    precision_t = precision[:-1]
    recall_t = recall[:-1]

    f1 = 2 * precision_t * recall_t / (precision_t + recall_t + 1e-12)

    best_idx = np.argmax(f1)

    return {
        "threshold": thresholds[best_idx],
        "precision": precision_t[best_idx],
        "recall": recall_t[best_idx],
        "f1": f1[best_idx]
    }

In [ ]:
threshold_result = find_best_threshold_by_f1(y_val, val_error)
threshold = threshold_result["threshold"]

In [ ]:
# Validation에서 선택한 threshold
print(f"threshold : {threshold:.8f}")
print(f"precision : {threshold_result['precision']:.4f}")
print(f"recall    : {threshold_result['recall']:.4f}")
print(f"f1        : {threshold_result['f1']:.4f}")

# Validation 성능 확인

In [ ]:
val_pred = (val_error >= threshold).astype(int)

In [ ]:
# Classification Report
print(classification_report(
    y_val,
    val_pred,
    target_names=["normal", "anomaly"],
    digits=4
))

In [ ]:
val_cm = confusion_matrix(y_val, val_pred)
plot_confusion_matrix(
    val_cm,
    title="Validation Confusion Matrix"
)

# 실제 application 예측 함수

In [ ]:
# 실제 운영에서는 label이 없습니다.
# 새 데이터 X_new만 들어옵니다.
#
# 함수 출력:
# - reconstruction_error
# - predicted_label
# - predicted_class

In [ ]:
def predict_new_connections(autoencoder, preprocessor, X_new, threshold, device):
    X_new_pre = preprocessor.transform(X_new).astype("float32")

    scores = reconstruction_error(
        model=autoencoder,
        X_array=X_new_pre,
        device=device
    )

    pred_label = (scores >= threshold).astype(int)
    pred_class = np.where(pred_label == 1, "anomaly", "normal")

    return pd.DataFrame({
        "reconstruction_error": scores,
        "predicted_label": pred_label,
        "predicted_class": pred_class
    })

# Test 데이터를 미래에 들어온 application 데이터라고 가정

In [ ]:
# 여기서는 y_test를 사용하지 않습니다.
# 실제 운영처럼 X_test만 보고 예측합니다.
application_predictions = predict_new_connections(
    autoencoder=autoencoder,
    preprocessor=preprocess,
    X_new=X_test,
    threshold=threshold,
    device=device
)

In [ ]:
# 미래 데이터 예측 결과 앞부분 20개
application_predictions.head(20)

In [ ]:
# 예측 결과 개수
print(application_predictions["predicted_class"].value_counts())

In [ ]:
# 원본 test 데이터와 예측 결과를 합칩니다.
app_result = X_test.reset_index(drop=True).copy()
app_result = pd.concat(
    [app_result, application_predictions.reset_index(drop=True)],
    axis=1
)

important_cols = [
    "predicted_class",
    "predicted_label",
    "reconstruction_error",
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "count",
    "srv_count",
    "serror_rate",
    "same_srv_rate",
    "dst_host_count",
    "dst_host_srv_count"
]

important_cols = [c for c in important_cols if c in app_result.columns]

In [ ]:
app_result.head()

In [ ]:
# Application 결과 예시
app_result[important_cols].head(20)

In [ ]:
# anomaly score가 높은 상위 alert
top_alerts = app_result.sort_values("reconstruction_error", ascending=False).head(20)
top_alerts[important_cols]

# Test label은 마지막 평가용으로만 사용

In [ ]:
# 실제 application에서는 y_test가 없습니다.
# 하지만 예제 데이터에는 정답 label이 있으므로 마지막에만 평가합니다.

test_error = application_predictions["reconstruction_error"].values
test_pred = application_predictions["predicted_label"].values

In [ ]:
# Test 성능 평가
# 실제 application에서는 이 단계의 정답 label이 없습니다.
# 여기서는 예제 검증을 위해서만 y_test를 사용합니다.

print("\nClassification Report")
print(classification_report(
    y_test,
    test_pred,
    target_names=["normal", "anomaly"],
    digits=4
))

In [ ]:
test_cm = confusion_matrix(y_test, test_pred)
plot_confusion_matrix(
    test_cm,
    title="Test Confusion Matrix"
)

# Encoder만 사용해서 latent feature 추출

In [ ]:
# 이 함수는 anomaly detection이 아니라 차원 축소 / feature extraction 용도입니다.
#
# 입력:
# - encoder: 학습된 encoder 모델
# - X_array: 전처리 완료된 numpy array
#
# 출력:
# - latent vector
# - shape: (sample 수, latent_dim)
def encode_features(encoder, X_array, device, batch_size=1024):
    encoder.eval()

    dataset = TensorDataset(torch.tensor(X_array, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    latent_vectors = []

    with torch.no_grad():
        for (batch_x,) in loader:
            batch_x = batch_x.to(device)

            z = encoder(batch_x)

            latent_vectors.append(z.cpu().numpy())

    return np.concatenate(latent_vectors, axis=0)

In [ ]:
Z_train_normal = encode_features(encoder=encoder, X_array=X_train_normal_pre, device=device)
Z_val = encode_features(encoder=encoder, X_array=X_val_pre, device=device)
Z_test = encode_features(encoder=encoder, X_array=X_test_pre, device=device)

In [ ]:
# Encoder latent vector 크기
print("Z_train_normal:", Z_train_normal.shape)
print("Z_val         :", Z_val.shape)
print("Z_test        :", Z_test.shape)

In [ ]:
latent_cols = [f"latent_{i}" for i in range(latent_dim)]
Z_test_df = pd.DataFrame(Z_test, columns=latent_cols)
Z_test_df.head(20)